In [8]:
### Importação de bibliotecas e leitura dos dados
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_excel(r"C:\Users\Usuário\Downloads\SCTEC\AULAS SCTEC\turma-visualizacao-de-dados\alunos\lourenco_lemos\semana_06\base\base_vendas_supermercado.xlsx")



c:\Users\Usuário\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\Usuário\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


In [18]:
### Passo 1 — Criando dados sujos

np.random.seed(42)
df_sujo = df.copy()

nan_config = {
    'Categoria':          10,
    'Quantidade':          8,
}

for coluna, qtd in nan_config.items(): 
    idx = np.random.choice(df_sujo.index, size=qtd, replace=False) 
    df_sujo.loc[idx, coluna] = np.nan

idx_out = np.random.choice(df_sujo.index, 3, replace=False) # Gerando 3 outliers
df_sujo.loc[idx_out, 'Valor Líquido'] = [5000, 4500, 0.01]

dup = df_sujo.sample(n=5, random_state=7)
df_sujo = pd.concat([df_sujo, dup], ignore_index=True)
df_sujo = df_sujo.sample(frac=1, random_state=3).reset_index(drop=True)

print(f"Base suja criada: {df_sujo.shape[0]} linhas | {df_sujo.isnull().sum().sum()} NaN | {df_sujo.duplicated().sum()} duplicatas")

Base suja criada: 255 linhas | 18 NaN | 5 duplicatas


In [19]:
# Passo 2 — Exibição de Diagnósticos
print("Dados sujos:")
for col in df_sujo.columns:
    n_missing = df_sujo[col].isna().sum()
    print(f"{col}: {n_missing} valores ausentes")
duplicados = df_sujo.duplicated().sum()
print(f"Duplicados: {duplicados}")

# Contando NaN por coluna
nan_por_coluna = df_sujo.isnull().sum()
pct_nan        = (df_sujo.isnull().mean() * 100).round(1)

print("=== Valores ausentes por coluna ===")
for col in df_sujo.columns:
    if nan_por_coluna[col] > 0:
        barras = '█' * int(pct_nan[col] )
        print(f"  {col:<25} {nan_por_coluna[col]:>3} NaN  ({pct_nan[col]:>4.1f}%)  {barras}")

print()
print(f"Total de NaN: {df_sujo.isnull().sum().sum()}")

Dados sujos:
Data: 0 valores ausentes
Loja: 0 valores ausentes
Cliente: 0 valores ausentes
Categoria: 10 valores ausentes
Produto: 0 valores ausentes
Quantidade: 8 valores ausentes
Preço Unitário: 0 valores ausentes
Desconto %: 0 valores ausentes
Valor Bruto: 0 valores ausentes
Valor Desconto: 0 valores ausentes
Valor Líquido: 0 valores ausentes
Forma de Pagamento: 0 valores ausentes
Duplicados: 5
=== Valores ausentes por coluna ===
  Categoria                  10 NaN  ( 3.9%)  ███
  Quantidade                  8 NaN  ( 3.1%)  ███

Total de NaN: 18


In [25]:
# Passo 3 - Limpeza dos dados
# removendo duplicatas
df_sem_dup = df_sujo.drop_duplicates()

print(f"Antes : {len(df_sujo)} linhas")
print(f"Depois: {len(df_sem_dup)} linhas")
print(f"Removidas: {len(df_sujo) - len(df_sem_dup)} duplicatas")
print(f"Duplicatas restantes: {df_sem_dup.duplicated().sum()}")

#preenchendo NaN de Categoria com a moda
moda_categoria = df_sem_dup['Categoria'].mode()[0]
df_sem_dup['Categoria'] = df_sem_dup['Categoria'].fillna(moda_categoria)

#preenchendo NaN de Quantidade com a mediana
mediana_quantidade = df_sem_dup['Quantidade'].median()
df_sem_dup['Quantidade'] = df_sem_dup['Quantidade'].fillna(mediana_quantidade)

print()

print("Limitando Valor Líquido entre 1 e 300")
#filtrar Valor Líquido entre 1 e 300
df_filtrada = df_sem_dup[(df_sem_dup['Valor Líquido'] >= 1) & (df_sem_dup['Valor Líquido'] <= 300)]
print(f"Antes : {len(df_sem_dup)} linhas")
print(f"Depois: {len(df_filtrada)} linhas")

Antes : 255 linhas
Depois: 250 linhas
Removidas: 5 duplicatas
Duplicatas restantes: 0

Limitando Valor Líquido entre 1 e 300
Antes : 250 linhas
Depois: 247 linhas


In [32]:
# Passo 4 - Transformações

df_nova = df_filtrada.copy()
df_nova['Mês']        = df_nova['Data'].dt.month
df_nova['Perfil_Compra'] = pd.cut(
    df_nova['Valor Líquido'],
    bins=[0, 120, 240, 300],
    labels=['Baixo', 'Médio', 'Alto']
)

print("Perfis de Compra:")
print(df_nova['Perfil_Compra'].value_counts().sort_index().to_string())


Perfis de Compra:
Perfil_Compra
Baixo    228
Médio     19
Alto       0


In [ ]:
#Passo 5 - Visualização

